In [7]:
# import required library and  set dataset path
import numpy as np
import pandas as pd
df=pd.read_csv(r"SMS SPAM DATASET\spam_sms.csv")

In [8]:
df

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


#### Data Cleaning

In [9]:
# rename column name
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

In [10]:
# remove missing and duplicate values
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [11]:
# Display dataset structure, column types, and non-null counts
print(df.info())

<class 'pandas.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   label   5169 non-null   str  
 1   text    5169 non-null   str  
dtypes: str(2)
memory usage: 121.1 KB
None


In [12]:
# top five rows
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [13]:
# bottom five rows
df.tail()

,label,text
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [14]:
# Import regex and NLTK text processing modules
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Download required NLTK datasets
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rehan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rehan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [15]:
# Initialize stemmer and load stop words
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()                                      # text convert into lowercase
    text = re.sub(r'[^a-zA-Z]', ' ', text)                   # revome special charecter etc.
    tokens = word_tokenize(text)                             # sentence convert into tokens
    # remove stopwords words make root form (length > 2)
    cleaned_tokens = [stemmer.stem(word) for word in tokens if word not in stop_words and len(word) > 2]
    return " ".join(cleaned_tokens)                          # add all words and make clean sentance
df['clean_text'] = df['text'].apply(clean_text)          # apply function on "text" dataframe



In [16]:
# chek clean_text
print(df[['text', 'clean_text']].head())

                                                text  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                          clean_text  
0  jurong point crazi avail bugi great world buff...  
1                                   lar joke wif oni  
2  free entri wkli comp win cup final tkt may tex...  
3                      dun say earli hor alreadi say  
4               nah think goe usf live around though  


#### Label Encoding

In [17]:
# Encode label column
df['label']=df['label'].map({'spam':1,'ham':0})

In [18]:
df

,label,text,clean_text
0,0,"Go until jurong point, crazy.. Available only ...",jurong point crazi avail bugi great world buff...
1,0,Ok lar... Joking wif u oni...,lar joke wif oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win cup final tkt may tex...
3,0,U dun say so early hor... U c already then say...,dun say earli hor alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though
...,...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...,time tri contact pound prize claim easi call p...
5568,0,Will Ì_ b going to esplanade fr home?,go esplanad home
5569,0,"Pity, * was in mood for that. So...any other s...",piti mood suggest
5570,0,The guy did some bitching but I acted like i'd...,guy bitch act like interest buy someth els nex...


#### Tokenization and Padding (Text to Numbers)

In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# initialize  Tokenizer( vocabulary size = 5000)
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")

# fit text data on clean_text
tokenizer.fit_on_texts(df['clean_text'])

# conver text into number(sequences)
sequences = tokenizer.texts_to_sequences(df['clean_text'])

# Padding to make same lenth of sequences (maxlen = 50)
max_len = 50
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

# check the shape
print("Padded sequences shape:", padded_sequences.shape)

Padded sequences shape: (5169, 50)


In [20]:
padded_sequences

array([[2851,  256,  504, ...,    0,    0,    0],
       [ 210,  505,  305, ...,    0,    0,    0],
       [   6,  370,  736, ...,    0,    0,    0],
       ...,
       [   1, 1310, 1179, ...,    0,    0,    0],
       [ 124, 1125, 1443, ...,    0,    0,    0],
       [1793,  353,  145, ...,    0,    0,    0]],
      shape=(5169, 50), dtype=int32)

#### Train-Test Split

In [21]:
from sklearn.model_selection import train_test_split

# Define feature variables and target labels
x = padded_sequences
y = df['label'].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [22]:
# chek shape train test data
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

x_train shape: (4135, 50)
x_test shape: (1034, 50)
y_train shape: (4135,)
y_test shape: (1034,)


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Create a sequential container to stack layers one by one
model = Sequential()

# Embedding Layer: Converts words into meaningful vector numbers
model.add(Embedding(input_dim=5000, output_dim=32, input_length=50))

# SimpleRNN Layer: Processes the sequence of text and understands context (32 units)
model.add(SimpleRNN(32, return_sequences=False))

# Output Layer: Gives the final prediction (0 or 1 / Spam or Ham) using sigmoid activation
model.add(Dense(1, activation='sigmoid'))

# Compile the model by setting the loss function and optimizer
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display the model summary and details
model.summary()

c:\Users\rehan\OneDrive\Desktop\Data Science\boss\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
# Train the model using training features and labels
model.fit(x_train, y_train, epochs=20, batch_size=32, validation_split=0.2)

Epoch 1/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.8721 - loss: 0.4034 - val_accuracy: 0.8924 - val_loss: 0.3512
Epoch 2/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8733 - loss: 0.3809 - val_accuracy: 0.8924 - val_loss: 0.3418
Epoch 3/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8733 - loss: 0.3810 - val_accuracy: 0.8924 - val_loss: 0.3430
Epoch 4/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8733 - loss: 0.3805 - val_accuracy: 0.8924 - val_loss: 0.3443
Epoch 5/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8791 - loss: 0.3646 - val_accuracy: 0.8924 - val_loss: 0.3445
Epoch 6/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8836 - loss: 0.3377 - val_accuracy: 0.9214 - val_loss: 0.2306
Epoch 7/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9453 - loss: 0.1701 - val_accuracy: 0.9069 - val_loss: 0.2586
Epoch 8/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9716 - loss: 0.1029 - val_accuracy: 0

save model and Tokenizer

In [25]:
# import pickle

# # Save the trained model
# model.save('spam_sms_model.h5')
# print("Model successfully saved as 'spam_sms_model.h5'")

# # Save the Tokenizer using pickle for later use
# with open('tokenizer.pkl', 'wb') as handle:
#     pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

Test model

In [26]:
# write your message
msg = ["Free entry to win cash prize"]

# Check prediction
pred = model.predict(pad_sequences(tokenizer.texts_to_sequences(msg), maxlen=50, padding='post'))
print("Spam" if pred[0][0] > 0.5 else "Ham")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step
Spam


In [27]:
# Evaluate on test data to check overall accuracy
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Test a few specific messages from x_test and compare with y_test
import numpy as np

# Let's check first 5 test messages
predictions = model.predict(x_test[:5])

for i in range(5):
    pred_label = 1 if predictions[i][0] > 0.5 else 0
    actual_label = y_test[i]

    print(f"\nMessage Index: {i}")
    print(f"Model Prediction: {'Spam' if pred_label == 1 else 'Ham'} (Score: {predictions[i][0]:.4f})")
    print(f"Actual Label (Asli Jawab): {'Spam' if actual_label == 1 else 'Ham'}")
    print("-" * 40)

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9400 - loss: 0.2721
Test Accuracy: 94.00%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step

Message Index: 0
Model Prediction: Ham (Score: 0.0030)
Actual Label (Asli Jawab): Ham
----------------------------------------

Message Index: 1
Model Prediction: Ham (Score: 0.0007)
Actual Label (Asli Jawab): Ham
----------------------------------------

Message Index: 2
Model Prediction: Ham (Score: 0.1190)
Actual Label (Asli Jawab): Ham
----------------------------------------

Message Index: 3
Model Prediction: Ham (Score: 0.0012)
Actual Label (Asli Jawab): Ham
----------------------------------------

Message Index: 4
Model Prediction: Ham (Score: 0.0009)
Actual Label (Asli Jawab): Ham
----------------------------------------
